# Analysis of pp Events: Event Activity Observables

Firstly, we load libraries and enable multi-threading, which allows event processing across multiple CPU cores.

In [1]:
import ROOT
import os

ROOT.EnableImplicitMT() # enable multi-threading

/home/javkovsky/anaconda3/envs/star-bachelor-env/lib/python3.11/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/javkovsky/anaconda3/envs/star-bachelor-env/include/site/python3.11); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


We connect RDataFrame to the TTree in our created events.root and events_smeared.root files.

In [2]:
# we must select the data we want to work with by their seed
seed = 22

# connect RDataFrame to the TTree in events_smeared.root
dfSmeared = ROOT.RDataFrame("events_smeared", f"data/{seed}/events{seed}_smeared.root")

We also make a file in which we will store the created histograms and response matrices for future unfolding, and a directory to store .pdf files in.

In [3]:
# create the directory for images
os.makedirs(f"img/{seed}", exist_ok=True)

# create the root file for histograms
file = ROOT.TFile(f"data/{seed}/events{seed}_plots.root", "RECREATE")

### Multiplicity

We plot the distributions of true and smeared multiplicity.

In [4]:
# create a data frame with multiplicity columns
dfNch = (
    dfSmeared.Define("NchTrue", "pT.size()") # define the true multiplicity
             .Define("NchSmeared", "pTSmeared.size()") # define the smeared multiplicity
)

# plot the distributions
for label in ["True", "Smeared"]:

    # load the histogram of the multiplicity distribution from the data frame
    histoNch = dfNch.Histo1D((f"histo{seed}Nch{label}", label + " Multiplicity;N_{ch};Normalized Events", 70, 0, 70), f"Nch{label}").GetValue()
    histoNch.Write() # store the histogram into the .root file -- note that we must save the histogram before normalizing it, as our response matrix is not constructed from the normalized distributions (and unfolding would thus fail)
    
    # set up canvas for drawing
    canvasNch = ROOT.TCanvas(f"canvas{seed}Nch{label}", f"{label} Multiplicity", 800, 600) # create a canvas to draw the histogram
    canvasNch.SetLogy() # set logarithmic scale for y-axis

    # normalize the histogram
    integral = histoNch.Integral()
    if integral > 0: # check if the histogram has entries to avoid division by zero
        histoNch.Scale(1.0 / integral) # normalize the histogram

    # customize the histogram
    histoNch.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
    histoNch.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

    # draw and save
    histoNch.Draw("HIST") # draw the histogram
    canvasNch.SaveAs(f"img/{seed}/{seed}Nch{label}.pdf") # save the canvas as a PDF file

Info in <TCanvas::Print>: pdf file img/22/22NchTrue.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22NchSmeared.pdf has been created


Then, we visualize the multiplicity response matrix.

In [5]:
# load the 2D histogram from the data frame
histoNchRM = dfNch.Histo2D((f"histo{seed}NchRM", "Multiplicity Response Matrix;N_{ch}^{MC};N_{ch}^{sm}", 70, 0, 70, 70, 0, 70), "NchTrue", "NchSmeared").GetValue() # 2D histogram with true Nch on the x-axis and smeared Nch on the y-axis

# set up canvas for drawing
canvasNchRM = ROOT.TCanvas(f"canvas{seed}NchRM", "Multiplicity Response Matrix", 800, 600)
canvasNchRM.SetLogz()

# draw and save
histoNchRM.Draw("COLZ")
canvasNchRM.SaveAs(f"img/{seed}/{seed}NchRM.pdf")

Info in <TCanvas::Print>: pdf file img/22/22NchRM.pdf has been created


As RooUnfold (which we are going to use for Bayesian unfolding) requires a response matrix with swapped axes, we store such histogram in the .root file. 

In [6]:
histoNchRMUnfolding = dfNch.Histo2D((f"histo{seed}NchRM", "Multiplicity Response Matrix;N_{ch}^{sm};N_{ch}^{MC}", 70, 0, 70, 70, 0, 70), "NchSmeared", "NchTrue").GetValue() # 2D histogram with smeared Nch on the x-axis and true Nch on the y-axis
histoNchRMUnfolding.Write()

4121

### Transverse Spherocities

In this section, we calculate the transverse spherocity $S_0$ and the unweighted transverse spherocity $S_0^{p_\mathrm{T} = 1}$ for events with $N_\mathrm{ch}(|\eta| < 1) > 10$. These observables are defined as
$$
S_0^{p_\mathrm{T} = 1} \coloneqq \frac{\pi^2}{4} \min_{\vec{n}} \left( \frac{\sum_i |\vec{u}_{\mathrm{T}, i} \times \vec{n}|}{N_\mathrm{trks}} \right)^2, \qquad 
S_0 \coloneqq \frac{\pi^2}{4} \min_{\vec{n}} \left( \frac{\sum_i |\vec{p}_{\mathrm{T}, i} \times \vec{n}|}{\sum_i |\vec{p}_{\mathrm{T}, i}|} \right)^2,
$$
where $\vec{u}_{\mathrm{T}, i}$ is the unit transverse momentum vector ($\vec{p}_{\mathrm{T}, i}$) of the $i$-th particle, $N_\mathrm{trks}$ is the number of particles entering the sum, and $\vec{n}$ is the unit vector which minimizes the quadratic expression.

Firstly, we define a data frame common for both regular and unweighted transverse spherocity observables.

In [7]:
# we need to load the calculateS0pT1(px, py, Nch) and calculateS0(ux, uy, px, py, pT, Nch) C++ functions
ROOT.gInterpreter.ProcessLine('#include "cpp/spherocities.cpp"')

# define new branches via the data frame
dfSpherocity = (
    
          # we apply the multiplicity cuts using the .Filter() before we define columns to prevent calculating the defined columns for events rejected by the filter
    dfNch.Filter("NchTrue > 10", "true multiplicity cut") # filter only events with true multiplicity greater than 10
         .Filter("NchSmeared > 10", "smeared multiplicity cut") # filter only events with smeared multiplicity greater than 10

          # define the true branches
         .Define("px", "pT * cos(phi)")
         .Define("py", "pT * sin(phi)")
         .Define("ux", "px / pT") 
         .Define("uy", "py / pT")
         .Define("S0pT1True", "calculateS0pT1(ux, uy, NchTrue)") # define the true unweighted spherocity column
         .Define("S0True", "calculateS0(ux, uy, px, py, pT, NchTrue)") # define the true spherocity column

          # define the smeared branches
         .Define("pxSmeared", "pTSmeared * cos(phiSmeared)")
         .Define("pySmeared", "pTSmeared * sin(phiSmeared)")
         .Define("uxSmeared", "pxSmeared / pTSmeared")
         .Define("uySmeared", "pySmeared / pTSmeared")
         .Define("S0pT1Smeared", "calculateS0pT1(uxSmeared, uySmeared, NchSmeared)") # define the true unweighted spherocity column
         .Define("S0Smeared", "calculateS0(uxSmeared, uySmeared, pxSmeared, pySmeared, pTSmeared, NchSmeared)") # define the true spherocity column
)

Afterwards, we plot the distributions of $S_0^{p_\mathrm{T} = 1}$ and $S_0$.

In [8]:
# plot the distributions
for abbrev, name, latex in zip(["S0", "S0pT1"], ["Transverse Spherocity", "Unweighted Transverse Spherocity"], ["S_{0}", "S_{0}^{p_{T} = 1}"]):

    for label in ["True", "Smeared"]:
        
        # load the histogram from the data frame
        histoSpherocity = dfSpherocity.Histo1D((f"histo{seed}{abbrev}{label}", label + " " + name + ";" + latex + ";Normalized Events", 100, 0, 1), abbrev + label).GetValue()
        histoSpherocity.Write() # store the histogram into the .root file

        # set up canvas for drawing
        canvasSpherocity = ROOT.TCanvas(f"canvas{seed}{abbrev}{label}", label + " " + name, 800, 600)
        canvasSpherocity.SetLogy()

        # normalize the histogram
        integral = histoSpherocity.Integral()
        if integral > 0:
            histoSpherocity.Scale(1.0 / integral)

        # customize the histogram
        histoSpherocity.SetLineColor(ROOT.kRed + 1)
        histoSpherocity.SetFillColorAlpha(ROOT.kRed, 0.1)

        # draw and save 
        histoSpherocity.Draw("HIST")
        canvasSpherocity.SaveAs(f"img/{seed}/{seed}{abbrev}{label}.pdf")

Info in <TCanvas::Print>: pdf file img/22/22S0True.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22S0Smeared.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22S0pT1True.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22S0pT1Smeared.pdf has been created


Finally, we plot the response matrices for both spherocities.

In [9]:
for abbrev, name, latex in zip(["S0", "S0pT1"], ["Transverse Spherocity", "Unweighted Transverse Spherocity"], ["S_{0}^{}", "S_{0}^{p_{T} = 1}"]):
    
    # load the 2D histogram from the data frame
    histoSpherocityRM = dfSpherocity.Histo2D((f"histo{seed}{abbrev}RM", f"{name} Response Matrix;" + latex[:-1] + " MC};" + latex[:-1] + " sm}", 100, 0, 1, 100, 0, 1), f"{abbrev}True", f"{abbrev}Smeared").GetValue() # 2D histogram with true unweighted spherocity on the x-axis and smeared unweighted spherocity on the y-axis

    # set up canvas for drawing
    canvasSpherocityRM = ROOT.TCanvas(f"canvas{seed}{abbrev}RM", f"{name} Response Matrix", 800, 600)
    canvasSpherocityRM.SetLogz()

    # draw and save
    histoSpherocityRM.Draw("COLZ")
    canvasSpherocityRM.SaveAs(f"img/{seed}/{seed}{abbrev}RM.pdf")

Info in <TCanvas::Print>: pdf file img/22/22S0RM.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22S0pT1RM.pdf has been created


Once again, we must construct and save a RooUnfold compatible response matrix. Additionally, we will save the true and smeared spherocity distribution here from the data frame with the double cut in true and smeared multiplicity: $N_\mathrm{ch}^\mathrm{MC} > 10 \quad \wedge \quad N_\mathrm{ch}^\mathrm{sm} > 10$.

In [10]:
for abbrev, name, latex in zip(["S0", "S0pT1"], ["Transverse Spherocity", "Unweighted Transverse Spherocity"], ["S_{0}^{}", "S_{0}^{p_{T} = 1}"]):
    histoSpherocityRMUnfolding = dfSpherocity.Histo2D((f"histo{seed}{abbrev}RM", f"{name} Response Matrix;" + latex[:-1] + " MC};" + latex[:-1] + " sm}", 100, 0, 1, 100, 0, 1), f"{abbrev}Smeared", f"{abbrev}True").GetValue() # 2D histogram with true unweighted spherocity on the y-axis and smeared unweighted spherocity on the x-axis
    histoSpherocityRMUnfolding.Write()

### $R_\mathrm{T}$

TODO

### $p_\mathrm{T}$ Spectra

In this section, we prepare 2D histograms of a given observable vs $p_\mathrm{T}$ (2D spectra) for unfolding. 

> **Note:** We use particle-level (or true) $p_\mathrm{T}$. If we were unfolding real experimental data, we would have to use $p_\mathrm{T}$ corrected for reconstruction efficiency (smearing) and detector acceptance.

Mapped onto **multiplicity** classes:

In [11]:
# create 2D histograms of multiplicity vs pT (0-10 GeV)
histopTNchTrue = dfNch.Histo2D((f"histo{seed}pTNchTrue", "True Multiplicity vs p_{T};p_{T}^{MC} [GeV];S_{0}^{p_{T}=1 MC}", 50, 0, 10, 70, 0, 70), "pT", "NchTrue")
histopTNchSmeared = dfNch.Histo2D((f"histo{seed}pTNchSmeared", "Smeared Multiplicity vs p_{T};p_{T}^{MC} [GeV];S_{0}^{p_{T}=1 sm}", 50, 0, 10, 70, 0, 70), "pT", "NchSmeared")

# dictionary for the for cycle
histospTNch = {
    "True": histopTNchTrue,
    "Smeared": histopTNchSmeared
}

for label, histo in histospTNch.items():
    # save the histogram for unfolding
    histo.Write() 

    # set up canvas
    canvaspTNch = ROOT.TCanvas(f"canvas{seed}pTNch{label}", label + " Multiplicity vs p_{T}", 800, 600)
    canvaspTNch.SetLogz()

    # draw and save
    histo.Draw("COLZ")
    canvaspTNch.SaveAs(f"img/{seed}/{seed}pTNch{label}.pdf")

Info in <TCanvas::Print>: pdf file img/22/22pTNchTrue.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTNchSmeared.pdf has been created


Mapped onto **spherocity** classes: 

In [12]:
for abbrev, name, latex in zip(["S0", "S0pT1"], ["Transverse Spherocity", "Unweighted Transverse Spherocity"], ["S_{0}^{}", "S_{0}^{p_{T} = 1}"]):

    # create 2D histograms of spherocity vs pT (0-10 GeV)
    histopTSpherocityTrue = dfSpherocity.Histo2D((f"histo{seed}pT{abbrev}True", f"True {name} vs " + "p_{T};p_{T}^{MC} [GeV];" + latex[:-1] + " MC}", 50, 0, 10, 100, 0, 1), "pT", f"{abbrev}True")
    histopTSpherocitySmeared = dfSpherocity.Histo2D((f"histo{seed}pT{abbrev}Smeared", f"Smeared {name} vs " + "p_{T};p_{T}^{MC} [GeV];" + latex[:-1] + " sm}", 50, 0, 10, 100, 0, 1), "pT", f"{abbrev}Smeared")

    # dictionary for the for cycle
    histospTSpherocity = {
        "True": histopTSpherocityTrue,
        "Smeared": histopTSpherocitySmeared
    }

    for label, histo in histospTSpherocity.items():
        # save the histogram for unfolding
        histo.Write() 

        # set up canvas
        canvaspTSpherocity = ROOT.TCanvas(f"canvas{seed}pT{abbrev}{label}", label + name + " vs p_{T}", 800, 600)
        canvaspTSpherocity.SetLogz()

        # draw and save
        histo.Draw("COLZ")
        canvaspTSpherocity.SaveAs(f"img/{seed}/{seed}pT{abbrev}{label}.pdf")

Info in <TCanvas::Print>: pdf file img/22/22pTS0True.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTS0Smeared.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTS0pT1True.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTS0pT1Smeared.pdf has been created


In the end, we close the .root file.

In [13]:
file.Close()